# Smart Traffic Monitoring — Ready to Run (2026)
**Session 07 · Video Processing · Google Colab**

**Instructions:** Open this notebook in Google Colab, select **Runtime → Run all**, upload an MP4 video when prompted, then wait for the annotated output. **No separate function copying is needed.**

This free OpenCV baseline detects **moving regions**, tracks their approximate centers, and counts crossings of one virtual line. It **does not classify cars vs. motorcycles** and may incorrectly count shadows, pedestrians or fragmented vehicles. For AI-based vehicle classification, extend this with YOLO + ByteTrack.

**Order:** 1. Imports → 2. Video upload → 3. Processing function → 4. Results. Run all cells in order. Recommended source: 10–20 seconds of roadside video from a relatively stationary camera.

## 1. Imports and video helpers

In [ ]:
import os
import cv2
import numpy as np
import subprocess
from IPython.display import Video, display

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def browser_mp4(input_path, output_path):
    """Create browser-compatible H.264 MP4 (requires ffmpeg, installed in Colab)."""
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(input_path),
        "-vcodec", "libx264", "-preset", "ultrafast", "-pix_fmt", "yuv420p",
        "-movflags", "+faststart", str(output_path)
    ], check=True)
    return str(output_path)


def show_video(video_path, width=640):
    display(Video(str(video_path), embed=True, width=width))

print("OpenCV version:", cv2.__version__)


## 2. Upload your traffic MP4
When prompted, choose an MP4 file **from your computer**. This cell does not download samples from GitHub.

If you already uploaded a file into Colab's left **Files** pane, you may instead set `traffic_input = '/content/your_filename.mp4'` and skip the upload command.

In [ ]:
if IN_COLAB:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Please choose an MP4 file, then rerun this cell.")
    traffic_input = next(iter(uploaded.keys()))
else:
    # Local execution (for testing only): set VIDEO_PATH in the environment.
    traffic_input = os.environ.get("VIDEO_PATH", "traffic_demo.mp4")

cap = cv2.VideoCapture(traffic_input)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {traffic_input!r}. Check that it is a valid MP4 file.")
ok, first_frame = cap.read()
if not ok:
    cap.release()
    raise RuntimeError("Video opened but has no readable frames. Please try another MP4 file.")
input_fps = cap.get(cv2.CAP_PROP_FPS) or 24
height, width = first_frame.shape[:2]
cap.release()
print(f"Ready: {traffic_input} | {width}×{height} | {input_fps:.1f} FPS")
show_video(traffic_input, width=640)


## 3. Complete traffic-tracking function
Run this cell **once**. Unlike previous snippets, it contains the full function and does not rely on missing functions. It tracks each moving object's center using distance and a short disappearance tolerance. IDs are approximate, especially under occlusion.

In [ ]:
def process_traffic(input_video, output_video="traffic_annotated_raw.mp4",
                    min_area=850, line_ratio=0.68, max_distance=90,
                    max_missing=7, max_frames=600):
    cap = cv2.VideoCapture(input_video)
    ok, first = cap.read()
    if not ok:
        cap.release()
        raise RuntimeError("Cannot read input video. Please upload a valid MP4 first.")
    height, width = first.shape[:2]
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    writer = cv2.VideoWriter(output_video,
                             cv2.VideoWriter_fourcc(*"mp4v"),
                             float(fps), (width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError("Cannot create output video. Check codec support.")
    subtractor = cv2.createBackgroundSubtractorMOG2(
        history=200, varThreshold=35, detectShadows=True)
    kernel = np.ones((5, 5), dtype=np.uint8)
    line_y = int(height * line_ratio)
    # id -> {center, previous_center, missing, counted}
    tracks = {}
    next_id = 1
    count = 0
    frame_count = 0
    try:
        while frame_count < max_frames:
            ok, frame = cap.read()
            if not ok:
                break
            frame_count += 1
            mask = subtractor.apply(frame)
            _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
            mask = cv2.dilate(mask, kernel, iterations=2)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
            detections = []
            for contour in contours:
                if cv2.contourArea(contour) < min_area:
                    continue
                x, y, w, h = cv2.boundingRect(contour)
                if w > width * .9 and h > height * .9:
                    continue  # background-model initialization
                detections.append((x, y, w, h))
            detections.sort(key=lambda d: d[2] * d[3], reverse=True)
            matched = set()
            for x, y, w, h in detections:
                center = (x + w // 2, y + h // 2)
                candidate, best_distance = None, float(max_distance)
                for oid, track in tracks.items():
                    if oid in matched or track["missing"] > max_missing:
                        continue
                    distance = float(np.hypot(center[0] - track["center"][0],
                                               center[1] - track["center"][1]))
                    if distance < best_distance:
                        best_distance = distance
                        candidate = oid
                if candidate is None:
                    oid = next_id
                    next_id += 1
                    tracks[oid] = {"center": center, "missing": 0, "counted": False}
                else:
                    oid = candidate
                    previous = tracks[oid]["center"]
                    # Crossing in either direction; count only once per tracked ID.
                    crossed = ((previous[1] < line_y <= center[1]) or
                               (previous[1] > line_y >= center[1]))
                    if crossed and not tracks[oid]["counted"]:
                        count += 1
                        tracks[oid]["counted"] = True
                    tracks[oid]["center"] = center
                    tracks[oid]["missing"] = 0
                matched.add(oid)
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 230, 90), 2)
                cv2.putText(frame, f"ID {oid}", (x, max(18, y-8)),
                            cv2.FONT_HERSHEY_SIMPLEX, .58, (0, 230, 90), 2)
            # Keep temporarily occluded IDs briefly rather than deleting immediately.
            for oid in list(tracks):
                if oid not in matched:
                    tracks[oid]["missing"] += 1
                    if tracks[oid]["missing"] > max_missing:
                        del tracks[oid]
            cv2.line(frame, (0, line_y), (width, line_y), (0, 255, 255), 2)
            cv2.rectangle(frame, (8, 8), (min(width-8, 490), 48), (25,25,25), -1)
            cv2.putText(frame, f"Moving objects crossed: {count}", (17, 36),
                        cv2.FONT_HERSHEY_SIMPLEX, .72, (255,255,255), 2)
            writer.write(frame)
    finally:
        cap.release()
        writer.release()
    if frame_count == 0:
        raise RuntimeError("Input video had no frames to process.")
    return output_video, count, frame_count

print("process_traffic() is ready.")


## 4. Process video and display results
If the boxes are too sensitive, increase `min_area` (e.g. 1800). If vehicles are not detected, reduce it (e.g. 500). The yellow line is at 68% of video height, and objects are counted only if they actually cross it.

In [ ]:
raw_output, traffic_count, processed_frames = process_traffic(
    input_video=traffic_input,
    output_video="traffic_annotated_raw.mp4",
    min_area=850,
    line_ratio=0.68,
    max_frames=600
)
result_path = browser_mp4(raw_output, "traffic_result.mp4")
print(f"Processed {processed_frames} frames")
print(f"Estimated moving-object crossings: {traffic_count}")
show_video(result_path, width=640)


## 5. Save result (optional)
The result is saved as `traffic_result.mp4` in the Colab file area. To download it onto your computer, run:

In [ ]:
if IN_COLAB:
    # Uncomment if you want the result to download automatically.
    # files.download(result_path)
    print("Result ready in Colab Files:", result_path)
else:
    print("Local output:", result_path)


## Student discussion (5–10 minutes)
1. Why does this algorithm sometimes count a shadow or a pedestrian as a vehicle?
2. What happens if the camera pans, zooms, or shakes?
3. Why can tracking IDs change when two vehicles overlap?
4. How would **YOLO detection + ByteTrack** improve this baseline, and what new limitations might remain?

**Instructor note:** This exercise is an **OpenCV motion-tracking baseline, not a trained vehicle classification model**. The count is based on apparent moving regions crossing a virtual line. In a true traffic system, configure class-specific detection, line-crossing directions, temporal tracking and precision/recall evaluation.